# HR Policy Assistant



### Import all packages


In [4]:
import os 
from dotenv import load_dotenv

# langchain framwork packages
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveJsonSplitter
from langchain_community.embeddings import JinaEmbeddings

In [5]:
load_dotenv()

True

### Load ENV Data

In [6]:
groq_api_key=os.getenv("GROQ_API_KEY")
jina_api_key=os.getenv("JINA_API_KEY")


# PHASE-1: DATA INGESTION PIPELINE


### Loading Data

In [ ]:
Data_Path=os.path.join("data","hr_policy.txt")

loader=TextLoader(Data_Path,encoding="utf-8")
documents=loader.load()
# print(documents[0].page_content)
# print(documents[0].metadata)
print(f"Loaded File : {Data_Path}")
print(f"Number of Document : {len(documents)}")
print(f"Total characters in document : {len(documents[0].page_content)}")
print("\n------------Previw of first 300 characters -------------")
print(documents[0].page_content[:300])

{'source': 'data\\hr_policy.txt'}
Loaded File : data\hr_policy.txt
Number of Document : 1
Total characters in document : 2598

------------Previw of first 300 characters -------------
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carr


#### LANGCHAIN DOCUMENT 

Langchain processes everything in form of documents 


DOCUMENTS : 

PAGE CONTENT -- the actual data 

METADATA  - extra information about the data like souce matlab kaha se data fetch kiya


In [13]:
len(documents)

1

In [14]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [15]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


### SPLITTING DATA

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks=text_splitter.split_documents(documents)
print(chunks)


[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [22]:
print(len(chunks))

9


In [24]:
print(chunks[2].page_content)

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.


### EMBEDD OUR DATA

In [25]:
from langchain_community.embeddings import JinaEmbeddings

embedding_model=JinaEmbeddings(
    model_name="jina-embeddings-v2-base-en"
)
print("EMB MODEL READY THE NAME IS ",embedding_model.model_name)

EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


### STRORE DATA IN VECTOR DB-FAISS

In [27]:
from langchain_community.vectorstores import FAISS

vector_store=FAISS.from_documents(chunks,embedding_model);

print("CHUNK ARE STORED", vector_store.index.ntotal)

CHUNK ARE STORED 9


### SEARCH QUERY

In [28]:
test_query="how many sick leaves employee get"

top_match=vector_store.similarity_search(test_query,k=2)
print(f"Query: {test_query}\n")

for i,match in enumerate(top_match,start=1):
    print(f"------Match {i}------")
    print(match.page_content)
    print()

Query: how many sick leaves employee get

------Match 1------
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

------Match 2------
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



## TOOL


In [30]:
# create a retriever object so that it can invke when needed
retriever=vector_store.as_retriever(search_kwargs={"k":3})# return top 3

def search_In_VectorsDB(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chucks=retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chucks)


# PHASE-2: DATA RETRIEVAL PIPELINE

### LLM SETUP

In [31]:
from langchain_groq import ChatGroq
llm=ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0 #creativity
)
print(llm.model)

openai/gpt-oss-120b


In [34]:
response=llm.invoke("what is ai in 2 line")
print(response.content)

Artificial Intelligence (AI) is the field of computer science that creates systems capable of performing tasks that normally require human intelligence—such as learning, reasoning, perception, and decision‑making. In essence, AI enables machines to mimic, augment, or surpass human cognitive abilities.


# AI AGENT
LLM=BRAIN

TOOL=RETRIEVER

MEMORY-NO MEMORY

In [35]:
from langchain.agents import create_agent

hr_assistant=create_agent(
    model=llm,
    tools=[search_In_VectorsDB],
    system_prompt=""" 
    
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)
print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [39]:
def ask_assistant(question:str)->str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)
    response=hr_assistant.invoke({"messages":[{
        "role":"user","content":question
    }]})

    answer=response["messages"][-1].content
    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer


In [46]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)
print(response["messages"][-1].content)

I’m part of the HR team at **Acme Corp**.


In [40]:
print(ask_assistant("tell me about sick leave policy"))

QUESTION: tell me about sick leave policy
------------------------------------------------------------
ANSWER: **Sick‑Leave Policy (Acme Crop)**  

- **Entitlement:** All full‑time employees receive **10 paid sick days per calendar year**.  
- **Separation from Annual Leave:** Sick leave is **tracked separately** from the 20 days of annual (vacation) leave.  
- **Medical Certificate:** If you are absent **more than 2 consecutive days** for illness, you must provide a **medical certificate** to HR.  
- **Usage:** Sick days can be taken as needed, subject to the above documentation requirement.  

If you need to request sick leave, log the request in the HR portal and attach the certificate (when required). Let me know if you’d like help with the portal steps or anything else!

**Sick‑Leave Policy (Acme Crop)**  

- **Entitlement:** All full‑time employees receive **10 paid sick days per calendar year**.  
- **Separation from Annual Leave:** Sick leave is **tracked separately** from the 